# Laboratorio 5: Clasificación de tweets sobre desastres

## Objetivo

El objetivo de este laboratorio es aplicar técnicas de minería de texto para analizar y clasificar tweets. Se busca construir un modelo que determine si un tweet se refiere a un desastre real.

La variable objetivo es `target`:

- `0`: el tweet no se refiere a un desastre real.
- `1`: el tweet se refiere a un desastre real.

A continuación, la descripción del conjunto de datos, limpieza y preprocesamiento del texto, análisis de unigramas y bigramas, y un modelo preliminar de clasificación.

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)
from wordcloud import WordCloud

In [ ]:
sns.set_theme(style="whitegrid")

nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

print("Librerías cargadas correctamente.")
print(f"Cantidad de stopwords en inglés: {len(stop_words)}")

Librerías cargadas correctamente.
Cantidad de stopwords en inglés: 198


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\maria\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
df = pd.read_csv("data/train.csv")

print("Dimensiones del dataset:", df.shape)
print("Columnas:", df.columns.tolist())

display(df.head())

# Información general
df.info()

Dimensiones del dataset: (7613, 5)
Columnas: ['id', 'keyword', 'location', 'text', 'target']


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


<class 'pandas.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        7613 non-null   int64
 1   keyword   7552 non-null   str  
 2   location  5080 non-null   str  
 3   text      7613 non-null   str  
 4   target    7613 non-null   int64
dtypes: int64(2), str(3)
memory usage: 297.5 KB


In [5]:
summary = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "valores_faltantes": df.isnull().sum(),
    "porcentaje_faltante": (df.isnull().mean() * 100).round(2),
    "valores_unicos": df.nunique()
})

display(summary)

print("Filas duplicadas completas:", df.duplicated().sum())
print("Textos duplicados:", df["text"].duplicated().sum())
print("IDs duplicados:", df["id"].duplicated().sum())

,tipo,valores_faltantes,porcentaje_faltante,valores_unicos
id,int64,0,0.00,7613
keyword,str,61,0.80,221
location,str,2533,33.27,3341
text,str,0,0.00,7503
target,int64,0,0.00,2


Filas duplicadas completas: 0
Textos duplicados: 110
IDs duplicados: 0


In [6]:
duplicated_text_rows = df[
    df.duplicated(subset="text", keep=False)
].sort_values("text")

print(
    "Cantidad total de filas involucradas en textos repetidos:",
    len(duplicated_text_rows)
)

print(
    "Cantidad de textos distintos que se repiten:",
    duplicated_text_rows["text"].nunique()
)

display(
    duplicated_text_rows[
        ["id", "keyword", "location", "text", "target"]
    ].head(20)
)

Cantidad total de filas involucradas en textos repetidos: 179
Cantidad de textos distintos que se repiten: 69


,id,keyword,location,text,target
4290,6094,hellfire,"Jubail IC, Saudi Arabia.",#Allah describes piling up #wealth thinking it...,0
4299,6105,hellfire,?????? ??? ?????? ????????,#Allah describes piling up #wealth thinking it...,0
4312,6123,hellfire,?????? ???? ??????,#Allah describes piling up #wealth thinking it...,1
6363,9095,suicide%20bomb,Nigeria,#Bestnaijamade: 16yr old PKK suicide bomber wh...,1
6373,9107,suicide%20bomb,Nigeria,#Bestnaijamade: 16yr old PKK suicide bomber wh...,1
6377,9113,suicide%20bomb,Nigeria,#Bestnaijamade: 16yr old PKK suicide bomber wh...,1
6378,9114,suicide%20bomb,Nigeria,#Bestnaijamade: 16yr old PKK suicide bomber wh...,1
6392,9135,suicide%20bomb,Nigeria,#Bestnaijamade: 16yr old PKK suicide bomber wh...,1
6366,9098,suicide%20bomb,Nigeria,#Bestnaijamade: 16yr old PKK suicide bomber wh...,1
2828,4064,displaced,NaN,#KCA #VoteJKT48ID 12News: UPDATE: A family of ...,1


In [7]:
targets_per_text = df.groupby("text")["target"].nunique()

conflicting_texts = targets_per_text[
    targets_per_text > 1
].index

conflicting_rows = (
    df[df["text"].isin(conflicting_texts)]
    .sort_values(["text", "target"])
)

print(
    "Textos repetidos con etiquetas contradictorias:",
    len(conflicting_texts)
)

print(
    "Filas involucradas en contradicciones:",
    len(conflicting_rows)
)

display(
    conflicting_rows[
        ["id", "keyword", "location", "text", "target"]
    ].head(30)
)

Textos repetidos con etiquetas contradictorias: 18
Filas involucradas en contradicciones: 55


,id,keyword,location,text,target
4290,6094,hellfire,"Jubail IC, Saudi Arabia.",#Allah describes piling up #wealth thinking it...,0
4299,6105,hellfire,?????? ??? ?????? ????????,#Allah describes piling up #wealth thinking it...,0
4312,6123,hellfire,?????? ???? ??????,#Allah describes piling up #wealth thinking it...,1
4244,6031,hazardous,"New Delhi, Delhi",#foodscare #offers2go #NestleIndia slips into ...,0
4221,5996,hazardous,NaN,#foodscare #offers2go #NestleIndia slips into ...,1
4239,6023,hazardous,"Mysore, Karnataka",#foodscare #offers2go #NestleIndia slips into ...,1
2832,4076,displaced,Pedophile hunting ground,.POTUS #StrategicPatience is a strategy for #G...,0
2830,4068,displaced,Pedophile hunting ground,.POTUS #StrategicPatience is a strategy for #G...,1
2831,4072,displaced,Pedophile hunting ground,.POTUS #StrategicPatience is a strategy for #G...,1
2833,4077,displaced,Pedophile hunting ground,.POTUS #StrategicPatience is a strategy for #G...,1


In [8]:
duplicate_summary = (
    duplicated_text_rows
    .groupby("target")
    .size()
    .rename("filas_involucradas")
    .reset_index()
)

duplicate_summary["categoría"] = duplicate_summary["target"].map({
    0: "No desastre",
    1: "Desastre real"
})

display(duplicate_summary)

,target,filas_involucradas,categoría
0,0,58,No desastre
1,1,121,Desastre real


## Descripción y calidad de los datos

El conjunto de datos contiene 7,613 registros y cinco variables. La columna `id` identifica cada tweet; `keyword` contiene una palabra clave asociada; `location` representa la ubicación declarada; `text` contiene el contenido del tweet; y `target` indica si el tweet se relaciona con un desastre real.

La variable `text`, que será la fuente principal de información para el modelo preliminar, no presenta valores faltantes. La columna `keyword` contiene 61 valores faltantes, equivalentes al 0.80 % de los registros, mientras que `location` contiene 2,533 valores faltantes, equivalentes al 33.27 %. Debido a que el modelo preliminar utilizará únicamente el texto, estos valores faltantes no requieren imputación por el momento. Tampoco se eliminarán filas debido a valores faltantes en `keyword` o `location`.

No se encontraron filas completas duplicadas ni identificadores repetidos. Sin embargo, se encontraron 110 repeticiones en la columna `text`. Esto significa que algunos tweets tienen el mismo contenido, pero poseen identificadores distintos o diferencias en otras variables.

Antes de eliminar estos registros es necesario determinar si los textos repetidos tienen la misma clasificación. Si un mismo texto aparece tanto con `target = 0` como con `target = 1`, existe una inconsistencia en las etiquetas. Además, permitir que textos idénticos aparezcan simultáneamente en entrenamiento y prueba podría causar fuga de información y producir métricas demasiado optimistas.

In [ ]:
# Eliminar todas las filas cuyos textos tienen etiquetas contradictorias
df_clean = df[
    ~df["text"].isin(conflicting_texts)
].copy()

# Conservar una sola aparición de cada texto restante
df_clean = (
    df_clean
    .drop_duplicates(subset="text", keep="first")
    .reset_index(drop=True)
)

print("Dimensiones originales:", df.shape)
print("Dimensiones después de tratar duplicados:", df_clean.shape)
print("Filas eliminadas:", len(df) - len(df_clean))

print(
    "Textos repetidos restantes:",
    df_clean["text"].duplicated().sum()
)

print(
    "Textos contradictorios restantes:",
    (df_clean.groupby("text")["target"].nunique() > 1).sum()
)